In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

# Подготовка числовых признаков
def prepare_num(df):
    df_num = df.drop(['Sex', 'Embarked', 'Pclass'], axis=1)
    df_sex = pd.get_dummies(df['Sex'])
    df_emb = pd.get_dummies(df['Embarked'], prefix='Emb')
    df_pcl = pd.get_dummies(df['Pclass'], prefix='Pclass')

    df_num = pd.concat((df_num, df_sex, df_emb, df_pcl), axis=1)

    return df_num


df_main = pd.read_csv('train.csv')

# Удаляем столбцы, которые не нужны как признаки
df_prep_x = df_main.drop(['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin'], axis=1)

# Отдельно сохраняем таргет — колонку Survived
df_prep_y = df_main['Survived']

# Применяем функцию, превращающую категориальные столбцы в one-hot
df_prep_x_num = prepare_num(df_prep_x)

# Заполняем пропуски медианами 
df_prep_x_num = df_prep_x_num.fillna(df_prep_x_num.median())


# Сначала делим данные на train (85%) и test (15%)
X_train, X_test, Y_train, Y_test = train_test_split(
    df_prep_x_num, df_prep_y, test_size=0.15, random_state=42
)

# Теперь делим train ещё раз: получаем train (≈72%) и validation (≈13%)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.15, random_state=42
)


df_main

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [ ]:
rf_parameters = {
     'n_estimators': [1, 3, 5, 7, 10],  # Количество деревьев в Random Forest
    'max_depth': [3, 5, 7, 9, 11, 13, 15], # Максимальная глубина дерева
    'min_samples_leaf': [2, 4, 6, 8],  # Минимальное количество объектов в конечном листе дерева
    'min_samples_split': [2, 4, 6, 8, 10]  # Минимальное количество объектов, чтобы узел начал делиться на ветки
}

rf_Gridd = GridSearchCV(RandomForestClassifier(random_state= 42), rf_parameters, cv= 5, scoring= 'accuracy')
rf_Gridd.fit(X_train, Y_train)
rf_bestt = rf_Gridd.best_estimator_


<class 'dict'>


In [3]:

#XGBoost
XGBoost_parameters = {
    'n_estimators': [100, 200],   # количество деревьев в бустинге    
    'max_depth': [3, 5, 7, 9],    # насколько глубокое каждое дерево      
    'learning_rate': [0.05, 0.1], # насколько сильно каждое новое дерево исправляет ошибки предыдущих     
}

XGBoost_Grid = GridSearchCV(XGBClassifier(random_state= 42), XGBoost_parameters, cv= 5, scoring= 'accuracy')
XGBoost_Grid.fit(X_train, Y_train)
XGBoost_best = XGBoost_Grid.best_estimator_

In [4]:
#Logistic Regression
logr_parameters = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000], # сила регуляризации: маленькое значение = модель проще, большое = модель гибче
    'solver': ['liblinear', 'lbfgs']
}

logr_Grid = GridSearchCV(LogisticRegression(max_iter= 1000,random_state= 42), logr_parameters, cv= 5, scoring= 'accuracy')
logr_Grid.fit(X_train, Y_train)
logr_best = logr_Grid.best_estimator_

In [5]:

#Knn
knn_parameters = {
    'n_neighbors': [3, 5, 7, 10],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_Grid = GridSearchCV(KNeighborsClassifier(), knn_parameters, cv= 5, scoring= 'accuracy')
knn_Grid.fit(X_train, Y_train)
knn_best = knn_Grid.best_estimator_

In [ ]:
X_train_full = pd.concat([X_train, X_val], axis=0)
Y_train_full = pd.concat([Y_train, Y_val], axis=0)

models = {
    'RandomForest': rf_bestt,
    'XGBoost': XGBoost_best,
    'LogRegression': logr_best,
    'KNN': knn_best
}

results = {}

for name, model in models.items():
    # дообучаем / переобучаем её на всём train_full (train + val)
    model.fit(X_train_full, Y_train_full)

    # получаем предсказания на ТЕСТОВОЙ части
    y_pred = model.predict(X_test)

    # считаем accuracy на тесте
    accuracy = accuracy_score(Y_test, y_pred)

    # сохраняем результат в словарь
    results[name] = accuracy

results

{'RandomForest': 0.8283582089552238,
 'XGBoost': 0.8134328358208955,
 'LogRegression': 0.7985074626865671,
 'KNN': 0.8059701492537313}

In [13]:
# Обучаем лучший RandomForest на полном тренировочном наборе (train + val)
rf_bestt.fit(X_train_full, Y_train_full)

# Получаем важности признаков, рассчитанные моделью
importances = rf_bestt.feature_importances_

# Сортируем индексы признаков по важности от наименьшей к наибольшей
indices = np.argsort(importances)

# Сохраняем список названий признаков
features = X_train.columns

features_num = [2, 4, 8]

results = {}

# Перебираем варианты количества признаков
for num in features_num:
    # Берём индексы num самых важных признаков с конца массива
    ind = indices[-num:]

    # Получаем соответствующие имена признаков
    feat = features[ind]

    # Оставляем только выбранные признаки в обучающей и тестовой выборках
    X_train_crop = X_train_full[feat]
    X_test_crop = X_test[feat]

    # Заново обучаем RandomForest, но уже только на отобранных признаках
    rf_bestt.fit(X_train_crop, Y_train_full)

    # Делаем предсказание на тестовой выборке
    y_pred = rf_bestt.predict(X_test_crop)

    accuracy = accuracy_score(Y_test, y_pred)
    results[str(num)] = accuracy

results


{'2': 0.7835820895522388, '4': 0.8208955223880597, '8': 0.8283582089552238}